[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Field Types and Defaults &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the `Mission` class its worked examples wrote. Run it
first. Task 4 empties `SQLModel.metadata` and builds a database of its own, so the tasks run in the
order they are written, and the last cell removes the scratch folder.


In [1]:
import re
import shutil
import subprocess
import sys
import uuid
from datetime import date, datetime, timezone
from decimal import Decimal
from enum import Enum
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from pydantic import ValidationError
from sqlalchemy import event, func, insert, text
from sqlalchemy.dialects import sqlite
from sqlalchemy.exc import IntegrityError, StatementError
from sqlalchemy.schema import CreateTable
from sqlmodel import Field, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


def table_sql(model):
    """The CREATE TABLE a table model describes, written for SQLite with no database anywhere."""
    return str(CreateTable(model.__table__).compile(dialect=sqlite.dialect())).strip()


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)


def mission_id(title):
    """The same id for the same title on every run: uuid4 would give a new one each time."""
    return uuid.uuid5(uuid.NAMESPACE_URL, f"https://example.edu/missions/{title}")


print("sqlmodel", sqlmodel.__version__, "| heroes:", len(HEROES), "| an id from a title:", mission_id("bridge"))


class Status(str, Enum):
    planned = "planned"
    running = "running"
    complete = "complete"


class Mission(SQLModel, table=True):
    id: uuid.UUID = Field(default_factory=uuid.uuid4, primary_key=True)
    hero_id: int | None = Field(default=None, foreign_key="hero.id")
    title: str = Field(unique=True, max_length=80)
    status: Status = Field(default=Status.planned, index=True)
    starts_on: date = Field(default_factory=date.today)              # Python fills this in
    filed_at: datetime | None = Field(default=None, sa_column_kwargs={"server_default": func.now()})
    reward: Decimal = Field(default=Decimal("0.00"), max_digits=8, decimal_places=2)
    secret: bool = False

SQLModel.metadata.create_all(engine)


sqlmodel 0.0.42 | heroes: 8 | an id from a title: f0180a0a-90b4-5aa6-9a38-7ee1dd172be0


**1.** A gadget, and the table it describes.


In [2]:
class Gadget(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=60)
    bought_on: date
    cost: Decimal = Field(default=Decimal("0.00"), max_digits=6, decimal_places=2)
    in_service: bool = True


SQLModel.metadata.create_all(engine)
print(table_sql(Gadget))


CREATE TABLE gadget (
	id INTEGER NOT NULL, 
	name VARCHAR(60) NOT NULL, 
	bought_on DATE NOT NULL, 
	cost NUMERIC(6, 2) NOT NULL, 
	in_service BOOLEAN NOT NULL, 
	PRIMARY KEY (id)
)


`in_service: bool = True` is a Python default, so the column is `BOOLEAN NOT NULL` with nothing after
it: the model fills it in, and the table does not.


**2.** Every column, with what it will accept.


In [3]:
for column in Gadget.__table__.columns:
    length = column.type.length if getattr(column.type, "length", None) else ""
    print(f"  {column.name:<12} {str(column.type):<14} nullable {str(column.nullable):<5} length {length}")


  id           INTEGER        nullable False length 
  name         VARCHAR(60)    nullable False length 60
  bought_on    DATE           nullable False length 
  cost         NUMERIC(6, 2)  nullable False length 
  in_service   BOOLEAN        nullable False length 


Only `NUMERIC(6, 2)` and `VARCHAR(60)` carry a size, and the `NUMERIC`'s is in `precision` and
`scale` rather than `length`, which is why the cost's length is blank.


**3.** Money, kept as money.


In [4]:
with Session(engine) as session:
    session.add(Gadget(name="Grapple line", bought_on=date(2026, 2, 14), cost=Decimal("249.99")))
    session.commit()

with Session(engine) as session:
    grapple = session.exec(select(Gadget)).one()
    print("cost back:", repr(grapple.cost), type(grapple.cost).__name__)

print("as a float  :", float("249.99") * 3)
print("as a Decimal:", Decimal("249.99") * 3)


cost back: Decimal('249.99') Decimal
as a float  : 749.97
as a Decimal: 749.97


The `Decimal` came back a `Decimal`, with its cents. The same sum in a `float` is off by a fraction
of a cent, which is invisible once and wrong by real money over a million rows.


**4.** The same model, with two more columns and two kinds of default.


In [5]:
class Condition(str, Enum):
    new = "new"
    worn = "worn"
    broken = "broken"


SQLModel.metadata.clear()                                           # every model's table, not only this one


class Gadget(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=60)
    bought_on: date
    cost: Decimal = Field(default=Decimal("0.00"), max_digits=6, decimal_places=2)
    in_service: bool = True
    condition: Condition = Field(default=Condition.new)
    checked_at: datetime | None = Field(default=None, sa_column_kwargs={"server_default": func.now()})


gadgets = hero_engine("scratch/gadgets.db")
SQLModel.metadata.create_all(gadgets)
for line in table_sql(Gadget).splitlines():
    if "condition" in line or "checked_at" in line:
        print(line.strip())


condition VARCHAR(6) NOT NULL,
checked_at DATETIME DEFAULT CURRENT_TIMESTAMP,


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sqlmodel/main.py:722: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.Gadget, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)


`Condition` is three values whose longest is `broken`, so the column is `VARCHAR(6)`. `checked_at`
carries its default into the table, which the next task depends on.


**5.** A row written as SQL, and what the table filled in.


In [6]:
with gadgets.begin() as connection:
    connection.execute(text("INSERT INTO gadget (name, bought_on, cost, in_service, condition) "
                            "VALUES ('Smoke pellet', '2026-03-09', 12.50, 1, 'worn')"))

with Session(gadgets) as session:
    pellet = session.exec(select(Gadget)).one()
    print("condition :", pellet.condition)
    print("checked_at:", pellet.checked_at is not None, type(pellet.checked_at).__name__)


condition : Condition.worn
checked_at: True datetime


`condition` had to be named, because its default is the model's and this row did not come from the
model. `checked_at` did not, because `CURRENT_TIMESTAMP` is in the table and the database filled it
in.


**6.** A gadget as a response would carry it.


In [7]:
def as_response(gadget):
    """The fields a response carries, as the types JSON has."""
    return {"name": gadget.name,
            "cost": f"{gadget.cost:.2f}",
            "bought_on": gadget.bought_on.isoformat()}


with Session(gadgets) as session:
    print(as_response(session.exec(select(Gadget)).one()))


{'name': 'Smoke pellet', 'cost': '12.50', 'bought_on': '2026-03-09'}


A `Decimal` and a `date` are Python types, and JSON has neither, so both are written as text. The
**Create, Read and Update Models** notebook is where a model does that conversion instead of a
function.

Last, the engines let go of their files, and this cell removes the scratch folder:


In [8]:
engine.dispose()
gadgets.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Field Types and Defaults](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/04-field-types-and-defaults.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
